# Установка зависимостей

In [ ]:
!pip install pymorphy3
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 11.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import pymorphy3
import nltk
import pandas as pd
import re
nltk.download('stopwords')
nltk.download('punkt')
morph = pymorphy3.MorphAnalyzer()
from nltk.corpus import stopwords
from nltk.stem.snowball import RussianStemmer # стемминг
from string import punctuation # убрать пунктуацию
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1.1 Создание базы данных

In [ ]:
path1 = '/content/drive/MyDrive/nlp/positive.csv'
path2 = '/content/drive/MyDrive/nlp/negative.csv'
data_positive = pd.read_csv(path1, delimiter = ';', header = None)
data_negative = pd.read_csv(path2, delimiter = ';', header = None)
shortened_data_positive = data_positive.sample(5000)
shortened_data_negative = data_negative.sample(5000)
df = pd.concat([shortened_data_negative, shortened_data_positive])

# Установить названия колонок
df.columns = ['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount']
df=df.sample(10000)
df=df.reset_index(drop=True)

In [ ]:
df.head()

,id,date,name,text,type,rep,rtw,fav,stcount,foll,frien,listcount
0,417648589956214784,1388410158,ksyuschenkakony,"Вчера играла на гитаре« реквием по мечте»,пале...",-1,0,0,0,62,3,16,0
1,413675603011727361,1387462924,StarkAurora,"RT @30SecondsAl: @StarkAurora ооо,чувак :(",-1,0,1,0,6948,1360,1352,2
2,418620526232616960,1388641885,ZloyTwit,"Бля не доехал до кожевниковой, уснул( сейчас с...",-1,0,0,0,21648,262,108,2
3,409756065404706816,1386528433,NsPani,"@DIRICFOREVER сладких снов,малыыш:*дадад,поско...",1,0,0,0,17596,2136,2054,16
4,410064640392318977,1386602003,Yrman2Yrman,"RT @TukvaSociopat: ""@lazarev_ua: На Майдане мн...",1,0,45,0,51821,188,131,7


# 1.2 Предобработка текста

In [ ]:
clean_texts = []

for k in range(len(df)):

  raw_text = df['text'][k]

  # удалить лишние символы
  raw_text = re.sub('[^А-Яа-яйЙёЁ]', ' ', raw_text)

  # разбить на слова с помощью регулярного выражения
  tokens = re.findall(r'\b\w+\b', raw_text)

  # убрать пунктуацию
  words_without_punkt = [i for i in tokens if i not in punctuation]

  # понизить регистр слов
  low_words = [i.lower() for i in words_without_punkt]

  # добавление в новый список только слов, не входящих в список стоп слов
  stopwords = nltk.corpus.stopwords.words('russian')

  # слово, которое нужно оставить в тексте
  word_to_keep = 'не'

  # убрать слово, которое нужно сохранить, из списка стоп слов
  if word_to_keep in stopwords:
      stopwords.remove(word_to_keep)

  words_without_stop = [i for i in low_words if i not in stopwords]

  # лемматизация
  lem_stem =[morph.parse(i)[0].normal_form for i in words_without_stop]

  # Объединить леммы в одну строку
  lem_text = ' '.join(lem_stem)

  # Добавить текст из лемм в список
  clean_texts.append(lem_text)

# Создать новый df для текстов из лемм
clean_df = pd.DataFrame(clean_texts, columns=['clean_text'])

clean_df.to_csv('/content/drive/MyDrive/nlp/clean_df.csv')

clean_df.head()

,clean_text
0,вчера играть гитара реквием мечта палец левый ...
1,ооо чувак
2,бля не доехать кожевников уснуть скоро попрать
3,сладкий сон малыыш дадада скорый мы ильгиз ещё...
4,майдан человек вромайдан


# 1.3 Тональный словарь LinisCrowd 2015

In [ ]:
sentiment_dict = pd.read_csv('/content/drive/MyDrive/nlp/words_all_full_rating.csv', encoding='windows-1251', delimiter=';')
sentiment_dict['mean'] = sentiment_dict['mean'].str.replace(',', '.').astype(float)
sentiment_dict = dict(zip(sentiment_dict['Words'], sentiment_dict['mean']))

clean_df['liniscrowd'] = clean_df['clean_text'].apply(lambda x: [round(sentiment_dict.get(x_split, 0), 2) for x_split in str(x).split(' ')])

In [ ]:
def liniscrowd_values(sentiment_scores):
  results = []
  if not sentiment_scores:
    sentiment_scores = [0]

  results.append([
      sum(sentiment_scores) / len(sentiment_scores),  # Среднее
      max(sentiment_scores),  # Максимальное
      min(sentiment_scores),  # Минимальное
      sum(sentiment_scores),  # Сумма
      sum(1 for s in sentiment_scores if s > 0),  # Количество положительных
      sum(1 for s in sentiment_scores if s < 0)   # Количество отрицательных
      ])

  return results

In [ ]:
clean_df['liniscrowd_values'] = clean_df['liniscrowd'].apply(liniscrowd_values)

In [ ]:
clean_df

,clean_text,liniscrowd,liniscrowd_values
0,вчера играть гитара реквием мечта палец левый ...,"[0, 0.0, 0, 0, 0.67, 0, 0.0, 0, 0, 0]","[[0.067, 0.67, 0, 0.67, 1, 0]]"
1,ооо чувак,"[0, 0]","[[0.0, 0, 0, 0, 0, 0]]"
2,бля не доехать кожевников уснуть скоро попрать,"[0, 0, 0, 0, 0.0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]"
3,сладкий сон малыыш дадада скорый мы ильгиз ещё...,"[0.89, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0]","[[0.08090909090909092, 0.89, 0, 0.89, 1, 0]]"
4,майдан человек вромайдан,"[0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]"
...,...,...,...
9995,наркота это зелёный чай прокатить хула,"[-1.67, 0, 0, 0, 0, 0]","[[-0.2783333333333333, 0, -1.67, -1.67, 0, 1]]"
9996,завтра поехать час пара последний день выходной,"[0, 0, 0.0, 0, 0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]"
9997,хотеть ранний не,"[0.0, 0.0, 0]","[[0.0, 0.0, 0.0, 0.0, 0, 0]]"
9998,сочувствовать это страшно именно поэтому стара...,"[1.0, 0, 0, 0, 0, 0, 0, 0, 0, -1.33, 0, 0, 0, 0]","[[-0.023571428571428577, 1.0, -1.33, -0.330000..."


# 1.4 PoS

In [ ]:
def get_pos_frequencies(text):
    words = text.split()
    pos_counts = Counter()

    for word in words:
        parsed_word = morph.parse(word)[0]
        pos_counts[parsed_word.tag.POS] += 1

    total_words = sum(pos_counts.values())
    if total_words > 0:
        pos_freq = {pos: count / total_words for pos, count in pos_counts.items()}
    else:
        pos_freq = {}
    return pos_freq

In [ ]:
clean_df['pos_frequencies'] = clean_df['clean_text'].apply(get_pos_frequencies)
df_pos = clean_df['pos_frequencies'].apply(pd.Series).fillna(0)
clean_df = pd.concat([clean_df, df_pos], axis=1).drop(columns=['pos_frequencies'])

In [ ]:
texts = clean_df['clean_text'].fillna('')

In [ ]:
clean_df.head(n=10)

,clean_text,liniscrowd,liniscrowd_values,ADVB,INFN,NOUN,ADJF,INTJ,PRCL,NPRO,...,CONJ,None,PRED,VERB,ADJS,PREP,GRND,PRTF,PRTS,COMP
0,вчера играть гитара реквием мечта палец левый ...,"[0, 0.0, 0, 0, 0.67, 0, 0.0, 0, 0, 0]","[[0.067, 0.67, 0, 0.67, 1, 0]]",0.100000,0.200000,0.600000,0.100000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ооо чувак,"[0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,бля не доехать кожевников уснуть скоро попрать,"[0, 0, 0, 0, 0.0, 0, 0]","[[0.0, 0, 0, 0.0, 0, 0]]",0.142857,0.428571,0.142857,0.000000,0.142857,0.142857,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,сладкий сон малыыш дадада скорый мы ильгиз ещё...,"[0.89, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0]","[[0.08090909090909092, 0.89, 0, 0.89, 1, 0]]",0.090909,0.181818,0.454545,0.181818,0.000000,0.000000,0.090909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,майдан человек вромайдан,"[0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,ахи согласный зомби апокалипсис,"[0, 0.67, -1.11, 0]","[[-0.11000000000000001, 0.67, -1.11, -0.440000...",0.000000,0.000000,0.750000,0.250000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,стыдно результат тест знание русский язык семь...,"[-1.67, 0.0, 0, 0, 0, -0.33, 0, 0.4, 0, 0]","[[-0.16, 0.4, -1.67, -1.6, 1, 2]]",0.100000,0.000000,0.500000,0.200000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,пойти отдохнуть минута проспать час,"[0.0, 0, 0, 0, 0.0]","[[0.0, 0.0, 0.0, 0.0, 0, 0]]",0.000000,0.600000,0.400000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,очень долго вспоминать звать новый секретарша ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0]","[[0.0, 0, 0, 0, 0, 0]]",0.222222,0.333333,0.222222,0.111111,0.000000,0.111111,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,добрый задерживаться,"[1.33, 0]","[[0.665, 1.33, 0, 1.33, 1, 0]]",0.000000,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 1.5 TF-IDF

In [ ]:
def extract_tfidf_features(texts, max_features=1000):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(texts)
    return tfidf_matrix.toarray()

In [ ]:
tfidf_features = extract_tfidf_features(clean_df["clean_text"])

In [ ]:
# Преобразуем tfidf_features в DataFrame
tfidf_df = pd.DataFrame(tfidf_features)

# Сохраняем DataFrame в CSV
tfidf_df.to_csv('tfidf_features.csv', index=False)

In [ ]:
tfidf_df.head(n=25)

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.46756,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.0,0.0


# 1.6 My Word2Vec

In [ ]:
from gensim.models import Word2Vec

In [ ]:
path1 = '/content/drive/MyDrive/nlp/positive.csv'
path2 = '/content/drive/MyDrive/nlp/negative.csv'
data_positive = pd.read_csv(path1, delimiter = ';', header = None)
data_negative = pd.read_csv(path2, delimiter = ';', header = None)
full_df = pd.concat([data_negative, data_positive])

# Установить названия колонок
full_df.columns = ['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount']
full_df = full_df.reset_index(drop=True)
full_df = full_df.drop(['id', 'date', 'name', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'], axis=1)

In [ ]:
stopwords = nltk.corpus.stopwords.words('russian')
# слово, которое нужно оставить в тексте
word_to_keep = 'не'

# убрать слово, которое нужно сохранить, из списка стоп слов
if word_to_keep in stopwords:
  stopwords.remove(word_to_keep)

# добавление в новый список только слов, не входящих в список стоп слов
stopwords = nltk.corpus.stopwords.words('russian')

# слово, которое нужно оставить в тексте
word_to_keep = 'не'

# убрать слово, которое нужно сохранить, из списка стоп слов
if word_to_keep in stopwords:
  stopwords.remove(word_to_keep)

words_without_stop = [i for i in low_words if i not in stopwords]

morph = MorphAnalyzer()
stemmer = RussianStemmer(True)

def preprocess(line, stop_words=stop):
  line = line.lower()
  line = re.sub('[^ЙйёЁА-Яа-я0-9 ]', '', line)
  line = re.sub('\s+', ' ', line).strip()
  line = [i for i in line.split() if i not in stopwords]
  line = [morph.parse(word)[0].normal_form for word in line]
  return ' '.join(line)

full_df['clean_text'] = full_df['text'].apply(preprocess)
full_df.to_csv('preprocessed.csv', index=False)

In [ ]:
full_df['tokens'] = full_df['clean_text'].apply(lambda x: str(x).split())

In [ ]:
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

In [ ]:
def get_w2v_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [ ]:
df['w2v_features'] = df['tokens'].apply(lambda x: get_w2v_vector(x, w2v_model))
df.to_csv('my_w2v.csv', index=False)

In [ ]:
joblib.dump(w2v_model, 'my_w2v_model.pkl')

In [ ]:
w2v_model_loaded['привет']

In [ ]:
pd.read_csv('/kaggle/input/nlp-1-lw/my_w2v.csv')['w2v_features']

# 1.7 W2V

In [ ]:
def get_w2v_vector2(tokens, model, vector_size=100):
    vectors = [model.get_vector(get_index(word)) for word in tokens if get_index(word) != 'NONE']
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)


w2v_model_loaded = KeyedVectors.load_word2vec_format("/kaggle/input/nlp-1-lw/model.bin", encoding='utf-8', unicode_errors='ignore', binary=True)
w2v_model_loaded.init_sims(replace=True)

In [ ]:
keys = list(w2v_model_loaded.key_to_index.keys())
values = [i.split('_')[0] for i in keys]
indexes = dict(zip(values, keys))

def get_index(word, indexes=indexes):
    word = str(word).lower()
    return indexes.get(word, 'NONE')


get_index('Интересно')

In [ ]:
df['w2v_features'] = df['tokens'].progress_apply(lambda x: get_w2v_vector2(x, w2v_model_loaded))

In [ ]:
df.to_csv('w2v.csv', index=False)

# My Fast Text

In [ ]:
corpus = df['tokens']
model = FastText(vector_size=100, window=5, min_count=1, workers=4)
model.build_vocab(corpus)  # Создание словаря
model.train(corpus, total_examples=len(corpus), epochs=10)  # Обучение
model.save("myfasttext.bin")

In [ ]:
def get_ft_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [ ]:
df['ft_features'] = df['tokens'].progress_apply(lambda x: get_ft_vector(x, model))
df.to_csv('my_ft.csv', index=False)

In [ ]:
FileLink('my_ft.csv')

# FT

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-ru-vectors", filename="model.bin")
model = fasttext.load_model(model_path)

In [ ]:
def get_ft_vector(tokens, model, vector_size=100):
    vectors = [model[word] for word in tokens]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [ ]:
df['ft_features'] = df['tokens'].progress_apply(lambda x: get_ft_vector(x, model))

In [ ]:
df.to_csv('ft.csv', index=False)

# Новый раздел

In [ ]:
path1 = '/content/drive/MyDrive/nlp/positive.csv'
path2 = '/content/drive/MyDrive/nlp/negative.csv'
data_positive = pd.read_csv(path1, delimiter = ';', header = None)
data_negative = pd.read_csv(path2, delimiter = ';', header = None)
full_df = pd.concat([data_negative, data_positive])

# Установить названия колонок
full_df.columns = ['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount']
full_df = full_df.reset_index(drop=True)
full_df = full_df.drop(['id', 'date', 'name', 'rep', 'rtw', 'fav', 'stcount', 'foll', 'frien', 'listcount'], axis=1)

In [ ]:
from pymorphy3 import MorphAnalyzer

In [ ]:
# добавление в новый список только слов, не входящих в список стоп слов
stop_words = nltk.corpus.stopwords.words('russian')
# слово, которое нужно оставить в тексте
word_to_keep = 'не'

# убрать слово, которое нужно сохранить, из списка стоп слов
if word_to_keep in stopwords:
  stop_words.remove(word_to_keep)

# слово, которое нужно оставить в тексте
word_to_keep = 'не'

# убрать слово, которое нужно сохранить, из списка стоп слов
if word_to_keep in stop_words:
  stop_words.remove(word_to_keep)

morph = MorphAnalyzer()
stemmer = RussianStemmer(True)

def preprocess(line, stopwords=stop_words):
  line = line.lower()
  line = re.sub('[^ЙйёЁА-Яа-я0-9 ]', '', line)
  line = re.sub('\s+', ' ', line).strip()
  line = [i for i in line.split() if i not in stopwords]
  line = [morph.parse(word)[0].normal_form for word in line]
  return ' '.join(line)

full_df['clean_text'] = full_df['text'].apply(preprocess)
full_df.to_csv('preprocessed.csv', index=False)

In [ ]:
!pip install spacy_udpipe
import spacy_udpipe
spacy_udpipe.download("ru")
nlp = spacy_udpipe.load("ru")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.8/936.8 kB 28.4 MB/s eta 0:00:00
Downloaded pre-trained UDPipe model for 'ru' language


In [ ]:
!pip install spacy

In [ ]:
import spacy

In [ ]:
!python -m spacy download ru_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.4/513.4 MB 2.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import re
import gensim
from gensim.models import Word2Vec, FastText
from sklearn.model_selection import train_test_split
#from ufal.udpipe import Model, Pipeline

# 2. Предобработка текста с UDPipe
texts_tokenized = full_df['clean_text']

# Загрузка модели для русского языка
nlp = spacy.load("ru_core_news_lg")

# Применяем spaCy для анализа текста
full_df['processed'] = full_df['clean_text'].apply(nlp)

# Функция для извлечения частей речи
def extract_pos_tags(doc):
    return ' '.join([f"{token.text}_{token.pos_}" for token in doc])

# Применяем функцию к каждому объекту Doc в столбце 'processed'
full_df['tagged_text'] = full_df['processed'].apply(extract_pos_tags)

# Извлекаем слова и их части речи
tagged_words = [f"{token.text}_{token.pos_}" for token in doc]

full_df[['clean_text', 'tagged_text']].head()

# 3. Обучение Word2Vec и FastText с нуля
w2v_model = Word2Vec(sentences=texts_tokenized, vector_size=100, window=5, min_count=2, workers=4)
fasttext_model = FastText(sentences=texts_tokenized, vector_size=100, window=5, min_count=2, workers=4)

# 4. Загрузка предобученных моделей Word2Vec и FastText
pretrained_w2v = gensim.models.KeyedVectors.load_word2vec_format("ruscorpora_upos_skipgram_300_5_2018.vec.gz", binary=False)
pretrained_fasttext = gensim.models.KeyedVectors.load_word2vec_format("araneum_none_fasttextcbow_300_5_2018.vec.gz", binary=False)

# 5. Функция для усреднения векторов слов в твите
def get_text_vector(text, model):
    vectors = [model[word] for word in text if word in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

# Генерация признаков
w2v_features = np.array([get_text_vector(text, w2v_model.wv) for text in texts_tokenized])
w2v_pretrained_features = np.array([get_text_vector(text, pretrained_w2v) for text in texts_tokenized])
fasttext_features = np.array([get_text_vector(text, fasttext_model.wv) for text in texts_tokenized])
fasttext_pretrained_features = np.array([get_text_vector(text, pretrained_fasttext) for text in texts_tokenized])

# 6. Разделение данных на train/test (только для обученных моделей)
X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(w2v_features, data['label'], test_size=0.2, random_state=42)
X_train_fasttext, X_test_fasttext, _, _ = train_test_split(fasttext_features, data['label'], test_size=0.2, random_state=42)

print(f"Размер train Word2Vec: {len(X_train_w2v)}, test Word2Vec: {len(X_test_w2v)}")
print(f"Размер train FastText: {len(X_train_fasttext)}, test FastText: {len(X_test_fasttext)}")

NameError: name 'doc' is not defined